# Funciones, sus gradientes, espacio de exploración y mínimos globales

Función de Rosenbrock:

$$ f(x,y) = (1-x)^2 + 100(y-x^2)^2 $$

Gradiente:

$$ \nabla f = [2(1-x) - 400x(y-x^2)]\hat{i} + 200(y-x^2)\hat{j} $$

Tiene mínimo global $0$ en $(1,1)$.

Función _Three-Hump Camel_:

$$ f(x,y) = 2x^2 - 1.05x^4 + \tfrac{x^6}{6} + xy + y^2 $$

Gradiente: 

$$ \nabla f = [4x - 4.20x^3 + x^5 + y]\hat{i} + (1 + 2y)\hat{j} $$

Mínimo global $0$ en $(0, 0)$, función definida en $x_i \in [-5, 5]$.

Función _Booth_:

$$ f(x,y) = (x+2y-7)^2 + (2x + y - 5)^2 $$

Gradiente: 

$$ \nabla f = [2(x + 2y - 7) + 4(2x + y - 5)]\hat{i} + [4(x + 2y - 7) + 2(2x + y - 5)]\hat{j} $$

$$ \nabla f = [10x + 8y - 34]\hat{i} + [8x + 10y - 38]\hat{j} $$

Mínimo global $0$ en $(1,3)$, función suele ser evaluada en $x_i \in [-10, 10]$.

Función Styblinski-Tang:

$$ f(x) = \tfrac{1}{2} \sum_{i=1}^d x_i^4 - 16x_i^2 + 5x_i $$

$$ \nabla f = \tfrac{1}{2} \sum_{i=1}^d 4x_i^3 - 32x_i + 5 $$

Yo consideré $ d=2 $ por simplicidad.

Mínimo global $-39.16599d$ en $(-2.903534, \dots, -2.903534)$.

Suele ser evaluada en el hípercubo $x_i \in [-5, 5]$

Vamos a explorar todas en el segmento $x,y \in [-5, 5]$

Importamos las bibliotecas de siempre

In [20]:
import torch
import sklearn
import matplotlib.pyplot as plt
import numpy as np 

In [21]:
DIMS = 2 # Vamos a usar 2 dimensiones para todas las funciones
torch.manual_seed(21) # Para resultados reproducibles

# Codificando las funciones (criterios (pérdida))

In [57]:
def rosenbrock(x, y):
    return (1-x)**2 + 100 * (y - x**2)**2

def three_hump_camel(x, y):
    return 2*x**2 - 1.05 * x**4 + (1 / 6)*x**6 + x*y + y**2

def booth(x, y):
    return (x + 2*y - 7)**2 + (2*x + y - 5)**2

def styblinski_tang(*args):
    xis = torch.stack(list(args))

    return 0.5*(xis**4 - 16*xis**2 + 5*xis).sum()

Ya que queremos comparar los diferentes métodos de descenso haremos que sea más o menos justo y le daremos el mismo punto de pártida a todos los métodos:

In [73]:
puntos_iniciales = np.random.normal(size=(DIMS,))
puntos_iniciales

array([-1.57605945,  0.72795352])

In [104]:
puntos_iniciales2 = np.random.normal(size=(DIMS,))
puntos_iniciales2

array([0.51535132, 0.35049691])

## Descenso de gradiente de una capa

In [95]:
def gd(punto_partida, criterio, lr=1e-3, n_epocas=100):
    print(f"Punto de partida: {punto_partida[0]}, {punto_partida[1]}")
    punto_actual = punto_partida.clone().detach().requires_grad_(True)
    historial = [punto_actual.detach()]
    for epoca in range(n_epocas):

        if punto_actual.grad is not None:
            punto_actual.grad.zero_()

        perdida = criterio(*punto_actual)
        perdida.backward()
        #optimizador.paso() no podemos utilizar el paso :c 
        # lo que vamos a hacer es hacer el descenso de gradiente manual con los valores de backward
        with torch.no_grad():
            punto_actual -= punto_actual.grad * lr
        

        historial.append(punto_actual.clone().detach())

        if epoca % 5 == 0 or epoca < 5:
            print(f"Pérdida {perdida.item()} en época {epoca + 1} en el punto ({punto_actual[0]}, {punto_actual[1]})")

    return punto_actual, historial

Ahora probaremos las funciones con lr, que parece funcionar más o menos bien 

### Descenso de gradiente en __Rosenbrock__

In [96]:
optimo, historial = gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), rosenbrock, 1e-3)

Punto de partida: -1.5760594606399536, 0.7279534935951233
Pérdida 314.99322509765625 en época 1 en el punto (-0.4638768434524536, 1.0791555643081665)
Pérdida 76.78801727294922 en época 2 en el punto (-0.6212601065635681, 0.9063608050346375)
Pérdida 29.709760665893555 en época 3 en el punto (-0.7473382949829102, 0.8022814393043518)
Pérdida 8.995421409606934 en época 4 en el punto (-0.8167141675949097, 0.7535280585289001)
Pérdida 4.048779487609863 en época 5 en el punto (-0.8413410186767578, 0.7362268567085266)
Pérdida 3.4710347652435303 en época 6 en el punto (-0.8472065925598145, 0.7305524349212646)
Pérdida 3.412217378616333 en época 11 en el punto (-0.8444908261299133, 0.7212522029876709)
Pérdida 3.3945419788360596 en época 16 en el punto (-0.8396908044815063, 0.7131664156913757)
Pérdida 3.376809597015381 en época 21 en el punto (-0.8348615765571594, 0.7050803899765015)
Pérdida 3.3590192794799805 en época 26 en el punto (-0.8300037980079651, 0.6969935894012451)
Pérdida 3.3411707878112

In [105]:
optimo, historial = gd(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), rosenbrock, 1e-3)

Punto de partida: 0.5153512954711914, 0.35049691796302795
Pérdida 0.9558542370796204 en época 1 en el punto (0.5338239669799805, 0.333514928817749)
Pérdida 0.45300036668777466 en época 2 en el punto (0.545122504234314, 0.3238055408000946)
Pérdida 0.27791979908943176 en época 3 en el punto (0.5518426299095154, 0.318476140499115)
Pérdida 0.22029370069503784 en época 4 en el punto (0.5558173060417175, 0.315686970949173)
Pérdida 0.20186004042625427 en época 5 en el punto (0.5582072734832764, 0.3143361508846283)
Pérdida 0.19593201577663422 en época 6 en el punto (0.5597028136253357, 0.3137879967689514)
Pérdida 0.19180099666118622 en época 11 en el punto (0.5629164576530457, 0.3148321211338043)
Pérdida 0.1901160329580307 en época 16 en el punto (0.5648941993713379, 0.31694748997688293)
Pérdida 0.18845795094966888 en época 21 en el punto (0.5667967796325684, 0.31910499930381775)
Pérdida 0.18682052195072174 en época 26 en el punto (0.5686811208724976, 0.3212546408176422)
Pérdida 0.185203284025

### Descenso de gradiente en __Three-Hump Camel__

In [81]:
optimo, historial = gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), three_hump_camel, 1e-3)

Punto de partida: -1.5760594606399536, 0.7279534935951233
Pérdida 0.4263404607772827 en época 1 en el punto (-1.577201247215271, 0.7280736565589905)
Pérdida 0.4250250458717346 en época 2 en el punto (-1.5783390998840332, 0.7281947135925293)
Pérdida 0.42371779680252075 en época 3 en el punto (-1.5794728994369507, 0.7283166646957397)
Pérdida 0.422419011592865 en época 4 en el punto (-1.5806026458740234, 0.7284395098686218)
Pérdida 0.42112988233566284 en época 5 en el punto (-1.5817283391952515, 0.7285632491111755)
Pérdida 0.4198499917984009 en época 6 en el punto (-1.5828499794006348, 0.7286878228187561)
Pérdida 0.4135841131210327 en época 11 en el punto (-1.5883954763412476, 0.7293238043785095)
Pérdida 0.4075484275817871 en época 16 en el punto (-1.5938336849212646, 0.7299808859825134)
Pérdida 0.4017449617385864 en época 21 en el punto (-1.5991610288619995, 0.7306582927703857)
Pérdida 0.3961770534515381 en época 26 en el punto (-1.6043740510940552, 0.7313552498817444)
Pérdida 0.39084440

In [107]:
optimo, historial = gd(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), three_hump_camel, 1e-3)

Punto de partida: 0.5153512954711914, 0.35049691796302795
Pérdida 0.763710081577301 en época 1 en el punto (0.5134779214859009, 0.34928056597709656)
Pérdida 0.7587264776229858 en época 2 en el punto (0.5116076469421387, 0.3480685353279114)
Pérdida 0.7537651062011719 en época 3 en el punto (0.5097405314445496, 0.34686079621315)
Pérdida 0.7488258481025696 en época 4 en el punto (0.5078765749931335, 0.3456573486328125)
Pérdida 0.7439088225364685 en época 5 en el punto (0.5060158371925354, 0.3444581627845764)
Pérdida 0.7390139102935791 en época 6 en el punto (0.5041583180427551, 0.3432632386684418)
Pérdida 0.7148730158805847 en época 11 en el punto (0.49492019414901733, 0.3373520076274872)
Pérdida 0.6912890672683716 en época 16 en el punto (0.48576676845550537, 0.33154553174972534)
Pérdida 0.6682630181312561 en época 21 en el punto (0.4767012596130371, 0.32584232091903687)
Pérdida 0.6457949876785278 en época 26 en el punto (0.46772661805152893, 0.32024088501930237)
Pérdida 0.62388437986373

### Descenso de gradiente en __Booth__

In [82]:
optimo, historial = gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), booth, 1e-3)

Punto de partida: -1.5760594606399536, 0.7279534935951233
Pérdida 105.81480407714844 en época 1 en el punto (-1.5321224927902222, 0.771282434463501)
Pérdida 102.04121398925781 en época 2 en el punto (-1.488971471786499, 0.8138265609741211)
Pérdida 98.40225219726562 en época 3 en el punto (-1.4465923309326172, 0.8556000590324402)
Pérdida 94.89309692382812 en época 4 en el punto (-1.4049712419509888, 0.8966168165206909)
Pérdida 91.5091552734375 en época 5 en el punto (-1.3640944957733154, 0.9368904232978821)
Pérdida 88.24591064453125 en época 6 en el punto (-1.323948621749878, 0.9764342904090881)
Pérdida 73.59503173828125 en época 11 en el punto (-1.1337319612503052, 1.1636593341827393)
Pérdida 61.37751007080078 en época 16 en el punto (-0.9599144458770752, 1.3345149755477905)
Pérdida 51.189178466796875 en época 21 en el punto (-0.8010736107826233, 1.4904234409332275)
Pérdida 42.69298553466797 en época 26 en el punto (-0.6559102535247803, 1.6326833963394165)
Pérdida 35.607872009277344 en

In [108]:
optimo, historial = gd(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), booth, 1e-3)

Punto de partida: 0.5153512954711914, 0.35049691796302795
Pérdida 46.54638671875 en época 1 en el punto (0.5413938164710999, 0.38086915016174316)
Pérdida 44.960025787353516 en época 2 en el punto (0.5669329166412354, 0.410729318857193)
Pérdida 43.42997360229492 en época 3 en el punto (0.5919777750968933, 0.44008657336235046)
Pérdida 41.95420837402344 en época 4 en el punto (0.61653733253479, 0.4689498841762543)
Pérdida 40.53079605102539 en época 5 en el punto (0.6406203508377075, 0.49732810258865356)
Pérdida 39.157875061035156 en época 6 en el punto (0.664235532283783, 0.5252298712730408)
Pérdida 32.98979949951172 en época 11 en el punto (0.775585412979126, 0.6578844785690308)
Pérdida 27.839563369750977 en época 16 en el punto (0.8764493465423584, 0.7798410058021545)
Pérdida 23.538253784179688 en época 21 en el punto (0.9677459597587585, 0.8920201063156128)
Pérdida 19.94498062133789 en época 26 en el punto (1.0503140687942505, 0.9952625036239624)
Pérdida 16.942276000976562 en época 31 

### Descenso de gradiente en __Styblinski-Tang__

In [83]:
optimo, historial = gd(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), styblinski_tang, 1e-3)

Punto de partida: -1.5760594606399536, 0.7279534935951233
Pérdida -23.005861282348633 en época 1 en el punto (-1.5959466695785522, 0.7363292574882507)
Pérdida -23.47215461730957 en época 2 en el punto (-1.615851879119873, 0.7448120713233948)
Pérdida -23.940906524658203 en época 3 en el punto (-1.6357675790786743, 0.7534027099609375)
Pérdida -24.411846160888672 en época 4 en el punto (-1.6556861400604248, 0.7621018886566162)
Pérdida -24.88471031188965 en época 5 en el punto (-1.6755996942520142, 0.7709102630615234)
Pérdida -25.359214782714844 en época 6 en el punto (-1.695500373840332, 0.7798285484313965)
Pérdida -27.7465877532959 en época 11 en el punto (-1.794529914855957, 0.8260894417762756)
Pérdida -30.132938385009766 en época 16 en el punto (-1.8920267820358276, 0.875180184841156)
Pérdida -32.484596252441406 en época 21 en el punto (-1.9869861602783203, 0.9271464347839355)
Pérdida -34.77196502685547 en época 26 en el punto (-2.0784554481506348, 0.9820054769515991)
Pérdida -36.97182

In [109]:
optimo, historial = gd(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), styblinski_tang, 1e-3)

Punto de partida: 0.5153512954711914, 0.35049691796302795
Pérdida -0.9000458121299744 en época 1 en el punto (0.5208231806755066, 0.3535187542438507)
Pérdida -0.9394038319587708 en época 2 en el punto (0.5263738036155701, 0.3565866947174072)
Pérdida -0.9799185991287231 en época 3 en el punto (0.5320041179656982, 0.35970139503479004)
Pérdida -1.0216212272644043 en época 4 en el punto (0.5377150177955627, 0.36286354064941406)
Pérdida -1.0645440816879272 en época 5 en el punto (0.5435075163841248, 0.3660737872123718)
Pérdida -1.1087201833724976 en época 6 en el punto (0.5493825078010559, 0.36933284997940063)
Pérdida -1.349609375 en época 11 en el punto (0.580028772354126, 0.38638490438461304)
Pérdida -1.627220869064331 en época 16 en el punto (0.6128811240196228, 0.404764860868454)
Pérdida -1.946548581123352 en época 21 en el punto (0.6480580568313599, 0.42456772923469543)
Pérdida -2.313076972961426 en época 26 en el punto (0.6856751441955566, 0.44589337706565857)
Pérdida -2.7327733039855

## Descenso de gradiente con momento

In [97]:
# este es casi lo mismo, solo necesita unos ajustes
def gd_con_momento(punto_partida, criterio, lr=1e-03, momento=0.9, n_epocas=100):
    punto_actual = punto_partida.clone().detach().requires_grad_(True)
    cambio = 0
    historial = [punto_actual.detach()]
    historial_derivadas = [0] # primero necesitamos un historial de derivadas
    historial_pasos = [torch.zeros_like(punto_actual.detach())] # tambien necesitamos un historial del tamaño de los pasos anteriores
    for epoca in range(n_epocas):

        if punto_actual.grad is not None:
            punto_actual.grad.zero_()

        perdida = criterio(*punto_actual)
        perdida.backward()
        #optimizador.paso() no podemos utilizar el paso :c 
        # lo que vamos a hacer es hacer el descenso de gradiente manual con los valores de backward
        with torch.no_grad():

            nuevo_cambio = lr*punto_actual.grad + momento*cambio
            punto_actual -= nuevo_cambio
            cambio = nuevo_cambio
        

        historial.append(punto_actual.clone().detach())

        if epoca % 5 == 0 or epoca < 5:
            print(f"Pérdida {perdida.item()} en época {epoca + 1} en el punto ({punto_actual[0]}, {punto_actual[1]})")

    return punto_actual, historial

### Descenso de gradiente con momento en __Rosenbrock__

In [99]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), rosenbrock, 0.9, 1e-5)

Pérdida 314.99322509765625 en época 1 en el punto (999.3883056640625, 316.8097839355469)
Pérdida 99692272680960.0 en época 2 en el punto (-359225786368.0, 179723152.0)
Pérdida inf en época 3 en el punto (1.6688026891523733e+37, 2.3227768896890584e+25)
Pérdida inf en época 4 en el punto (-inf, inf)
Pérdida nan en época 5 en el punto (nan, nan)
Pérdida nan en época 6 en el punto (nan, nan)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en época 61 en el punto (nan, nan)
Pérdida nan en época 66 en el punto (nan, nan)
Pérdida nan en época 71 en el punto (nan, nan

In [110]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), rosenbrock, 0.9, 1e-5)

Pérdida 0.9558542370796204 en época 1 en el punto (17.140758514404297, -14.933292388916016)
Pérdida 9532231.0 en época 2 en el punto (-1905138.625, 55558.06640625)
Pérdida 1.3173656687347174e+27 en época 3 en el punto (2.4893286652320227e+21, 653319622098944.0)
Pérdida inf en época 4 en el punto (-inf, inf)
Pérdida nan en época 5 en el punto (nan, nan)
Pérdida nan en época 6 en el punto (nan, nan)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en época 61 en el punto (nan, nan)
Pérdida nan en época 66 en el punto (nan, nan)
Pérdida nan en época 71 en el punto

### Descenso de gradiente con momento en __Three-Hump Camel__

In [101]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), three_hump_camel, 0.9, 1e-3)

Pérdida 0.4263404607772827 en época 1 en el punto (-2.6036624908447266, 0.8360906839370728)
Pérdida 15.749509811401367 en época 2 en el punto (46.9852294921875, 1.6745318174362183)
Pérdida 1788038272.0 en época 3 en el punto (-205694432.0, -43.625492095947266)
Pérdida inf en época 4 en el punto (inf, 185125008.0)
Pérdida nan en época 5 en el punto (nan, -inf)
Pérdida nan en época 6 en el punto (nan, nan)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en época 61 en el punto (nan, nan)
Pérdida nan en época 66 en el punto (nan, nan)
Pérdida nan en época 71 en e

In [111]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), three_hump_camel, 0.9, 1e-3)

Pérdida 0.763710081577301 en época 1 en el punto (-1.1707056760787964, -0.7442137002944946)
Pérdida 2.622957229614258 en época 2 en el punto (-0.37395602464675903, 1.6479111909866333)
Pérdida 2.3589730262756348 en época 3 en el punto (-0.7011315822601318, -0.9793764352798462)
Pérdida 2.395081043243408 en época 4 en el punto (1.5537054538726807, 1.411892056465149)
Pérdida 5.240893363952637 en época 5 en el punto (0.7207041382789612, -2.5254569053649902)
Pérdida 5.336729049682617 en época 6 en el punto (1.6382725238800049, 1.3677945137023926)
Pérdida 5.794701099395752 en época 11 en el punto (0.6133878827095032, -2.562943935394287)
Pérdida 5.5227484703063965 en época 16 en el punto (1.5599864721298218, 1.4433395862579346)
Pérdida 5.533734321594238 en época 21 en el punto (0.6811803579330444, -2.5589401721954346)
Pérdida 5.5547637939453125 en época 26 en el punto (1.5821211338043213, 1.4426233768463135)
Pérdida 5.431642055511475 en época 31 en el punto (0.6217730045318604, -2.581884622573

### Descenso de gradiente con momento en __Booth__

In [102]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), booth, 0.9, 1e-5)

Pérdida 105.81480407714844 en época 1 en el punto (37.96720886230469, 39.724002838134766)
Pérdida 24436.8046875 en época 2 en el punto (-559.1500854492188, -556.9555053710938)
Pérdida 5645864.0 en época 3 en el punto (8513.8740234375, 8515.71875)
Pérdida 1304418688.0 en época 4 en el punto (-129393.484375, -129391.359375)
Pérdida 301372473344.0 en época 5 en el punto (1966794.625, 1966796.75)
Pérdida 69628990586880.0 en época 6 en el punto (-29895244.0, -29895242.0)
Pérdida 4.5837728664636525e+25 en época 11 en el punto (24255966216192.0, 24255966216192.0)
Pérdida 3.0175603636145253e+37 en época 16 en el punto (-1.9680448896632357e+19, -1.9680448896632357e+19)
Pérdida inf en época 21 en el punto (1.5968030861173602e+25, 1.5968030861173602e+25)
Pérdida inf en época 26 en el punto (-1.295590205737925e+31, -1.295590205737925e+31)
Pérdida inf en época 31 en el punto (1.0511969150856488e+37, 1.0511969150856488e+37)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el

In [113]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), booth, 0.9, 1e-5)

Pérdida 46.54638671875 en época 1 en el punto (23.95361328125, 27.685495376586914)
Pérdida 10214.1796875 en época 2 en el punto (-360.36419677734375, -359.7496643066406)
Pérdida 2359535.0 en época 3 en el punto (5503.70703125, 5506.8154296875)
Pérdida 545145984.0 en época 4 en el punto (-83648.0625, -83646.953125)
Pérdida 125950328832.0 en época 5 en el punto (1271472.25, 1271474.875)
Pérdida 29099519115264.0 en época 6 en el punto (-19326352.0, -19326350.0)
Pérdida 1.9156609373311033e+25 en época 11 en el punto (15680732659712.0, 15680732659712.0)
Pérdida 1.261105720179251e+37 en época 16 en el punto (-1.2722801988228612e+19, -1.2722801988228612e+19)
Pérdida inf en época 21 en el punto (1.0322836030057517e+25, 1.0322836030057517e+25)
Pérdida inf en época 26 en el punto (-8.375590402943422e+30, -8.375590402943422e+30)
Pérdida inf en época 31 en el punto (6.7956619025227e+36, 6.7956619025227e+36)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, na

### Descenso de gradiente con momento en __Styblinski-Tang__

In [103]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales, requires_grad=True, dtype=torch.float32), styblinski_tang, 0.9, 1e-5)

Pérdida -23.005861282348633 en época 1 en el punto (-19.47454261779785, 8.266125679016113)
Pérdida 70643.9453125 en época 2 en el punto (12992.412109375, -891.6181640625)
Pérdida 1.4247502811234304e+16 en época 3 en el punto (-3947678924800.0, 1275864576.0)
Pérdida inf en época 4 en el punto (1.1073832588461907e+38, -3.7384038723222447e+27)
Pérdida nan en época 5 en el punto (nan, inf)
Pérdida nan en época 6 en el punto (nan, nan)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en época 61 en el punto (nan, nan)
Pérdida nan en época 66 en el punto (nan, nan)
P

In [114]:
optimo, historial = gd_con_momento(torch.tensor(puntos_iniciales2, requires_grad=True, dtype=torch.float32), styblinski_tang, 0.9, 1e-5)

Pérdida -0.9000458121299744 en época 1 en el punto (5.440042495727539, 3.070148229598999)
Pérdida 191.44371032714844 en época 2 en el punto (-208.2606201171875, -7.059228897094727)
Pérdida 940239552.0 en época 3 en el punto (16255795.0, 522.2427368164062)
Pérdida 3.4914261713034985e+28 en época 4 en el punto (-7.732093966343677e+21, -256375248.0)
Pérdida nan en época 5 en el punto (inf, 3.0331981280400616e+25)
Pérdida nan en época 6 en el punto (nan, -inf)
Pérdida nan en época 11 en el punto (nan, nan)
Pérdida nan en época 16 en el punto (nan, nan)
Pérdida nan en época 21 en el punto (nan, nan)
Pérdida nan en época 26 en el punto (nan, nan)
Pérdida nan en época 31 en el punto (nan, nan)
Pérdida nan en época 36 en el punto (nan, nan)
Pérdida nan en época 41 en el punto (nan, nan)
Pérdida nan en época 46 en el punto (nan, nan)
Pérdida nan en época 51 en el punto (nan, nan)
Pérdida nan en época 56 en el punto (nan, nan)
Pérdida nan en época 61 en el punto (nan, nan)
Pérdida nan en época 6

## Descenso de gradiente con momento de una capa

## Descenso ADAM de una capa